# 01 - EDA y Modelo Base | Clasificador de Sentimiento (Olist)

Continuación del Proyecto 2. Objetivo: clasificar reseñas de clientes en **positivas (4-5★)** y **negativas (1-2★)**, descartando las neutrales (3★).

**Decisiones de diseño:**
- Etiquetado **binario**, se descartan las 3★.
- Solo se usan reseñas **con texto** (`review_comment_message` no nulo).
- Idioma: portugués brasileño.

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.max_colwidth', 120)

# Ruta local (repo clonado). En Colab no existe, así que se descarga de Kaggle.
RAW = '../data/raw/olist_order_reviews_dataset.csv'

if not os.path.exists(RAW):
    import kagglehub
    ruta = kagglehub.dataset_download('olistbr/brazilian-ecommerce')
    RAW = os.path.join(ruta, 'olist_order_reviews_dataset.csv')

print('Dataset:', RAW)

## 1. Carga y vista general

In [ ]:
df = pd.read_csv(RAW)
print('Filas totales:', len(df))
df.head()

## 2. Calidad de datos: ¿cuántas reseñas tienen texto?

Hallazgo clave de auditoría: la mayoría de las reseñas solo traen estrellas, sin comentario. Para un clasificador de texto solo sirven las filas con comentario.

In [ ]:
print('Con comentario:', df['review_comment_message'].notna().sum())
print('Sin comentario:', df['review_comment_message'].isna().sum())
print('\nDistribución de estrellas (todas):')
print(df['review_score'].value_counts().sort_index())

## 3. Etiquetado: binario descartando 3★

In [ ]:
# Solo filas con texto
txt = df[df['review_comment_message'].notna()].copy()

# Descartar neutrales (3★)
txt = txt[txt['review_score'] != 3].copy()

# Etiqueta binaria: 1 = positivo (4-5), 0 = negativo (1-2)
txt['sentimiento'] = (txt['review_score'] >= 4).astype(int)

print('Total utilizable:', len(txt))
print(txt['sentimiento'].value_counts(normalize=True).rename({0: 'negativo', 1: 'positivo'}))

## 4. Exploración del texto
_(longitud de comentarios, palabras frecuentes por clase, etc.)_

In [ ]:
txt['n_chars'] = txt['review_comment_message'].str.len()
txt.groupby('sentimiento')['n_chars'].describe()

## 5. Baseline tonto (clase mayoritaria)

Vara a superar: un modelo que siempre predice 'positivo'. Si el modelo real no la supera con claridad, no aprendió nada útil.

In [ ]:
baseline_acc = txt['sentimiento'].mean()  # proporción de positivos
print(f'Accuracy del baseline (siempre positivo): {baseline_acc:.3f}')

## 6. Modelo base: TF-IDF + Regresión Logística

Se usa `Pipeline` para evitar *data leakage*: el vectorizador se ajusta SOLO con el conjunto de entrenamiento.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, confusion_matrix

X = txt['review_comment_message']
y = txt['sentimiento']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

pipe = Pipeline([
    ('tfidf', TfidfVectorizer(min_df=5, ngram_range=(1, 2))),
    ('clf', LogisticRegression(max_iter=1000, class_weight='balanced')),
])

pipe.fit(X_train, y_train)
y_pred = pipe.predict(X_test)

print(classification_report(y_test, y_pred, target_names=['negativo', 'positivo']))
print(confusion_matrix(y_test, y_pred))

## 7. Análisis de errores _(sección estrella)_

Revisar reseñas mal clasificadas y agruparlas por patrón: sarcasmo, reseñas mixtas, portugués coloquial, etc. Documentar *dónde y por qué* falla el modelo lineal.

In [ ]:
errores = X_test.to_frame()
errores['real'] = y_test.values
errores['pred'] = y_pred
errores = errores[errores['real'] != errores['pred']]
print('Total mal clasificadas:', len(errores))
errores.head(20)